# 00 — Environment and harness check

Run this first. It checks the local StateManagement package before any Gemini request. It does not make network calls or create experimental results.

In [ ]:
from pathlib import Path
import os, sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
if not (PROJECT_ROOT / 'src' / 'state_management').is_dir():
    raise RuntimeError('Open this notebook from the StateManagement project or its experiments folder.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Python: {sys.version.split()[0]}')
print(f'Project: {PROJECT_ROOT}')
print(f'Gemini key configured: {bool(os.environ.get("GOOGLE_API_KEY"))}')
print('A key is not needed for this notebook.')

In [ ]:
from state_management.models import MemoryMode, SelectionStrategy

print('Memory modes:', [mode.value for mode in MemoryMode])
print('Retrieval strategies:', [strategy.value for strategy in SelectionStrategy])
assert MemoryMode.NONE.value == 'none'
assert SelectionStrategy.RELEVANCE.value == 'relevance'

In [ ]:
import subprocess

environment = os.environ.copy()
environment['PYTHONPATH'] = str(PROJECT_ROOT / 'src') + os.pathsep + environment.get('PYTHONPATH', '')
test = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=False)
print(test.stdout)
if test.returncode:
    print(test.stderr)
    raise RuntimeError('Unit tests failed. Fix these before continuing.')
print('Harness unit tests passed.')

In [ ]:
demo = subprocess.run([sys.executable, '-m', 'state_management', 'demo', '--strategy', 'relevance', '--top-k', '1'], cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=True)
print(demo.stdout)
print('This deterministic demo is a pipeline check, not a Gemini result.')